# 02 — Feature Engineering

Мета: створити стабільний пайплайн фічей з SQLite бази та зберегти processed-датасет.

**Фічі (~35 штук):**
- Часові: instant registration, нічні транзакції, активні дні, velocity
- Гео: country mismatch (card/payment), кількість країн
- Суми: avg, max, std, total, range
- Помилки: fraud/antifraud errors, error rate, error types
- Карти/платежі: brands, currencies, transaction types
- Профіль: gender, name-email match, traffic type

In [ ]:
import sys
sys.path.insert(0, '../..')

from src.features import load_data_sqlite, load_data_csv, extract_features, apply_target_encoding, setup_directories

## 1. Завантаження даних

In [ ]:
setup_directories()

# Train з SQLite
train_users, train_tx = load_data_sqlite('../../data/sql/train_data.db')

# Test з CSV
test_users, test_tx = load_data_csv()

## 2. Генерація фічей

In [ ]:
train_features = extract_features(train_users, train_tx)
test_features = extract_features(test_users, test_tx)

print(f"Train features: {train_features.shape}")
print(f"Test features:  {test_features.shape}")
train_features.head()

## 3. Target Encoding
Кодуємо категоріальну змінну `traffic_type` середнім значенням таргету.
Навчаємося **тільки на train** для уникнення data leakage.

In [ ]:
train_features, test_features = apply_target_encoding(train_features, test_features)

print(f"\nFinal train shape: {train_features.shape}")
print(f"Final test shape:  {test_features.shape}")
print(f"\nFeature columns: {[c for c in train_features.columns if c not in ['id_user', 'is_fraud']]}")

## 4. Збереження

In [ ]:
train_features.to_csv('../../data/processed/train_features.csv', index=False)
test_features.to_csv('../../data/processed/test_features.csv', index=False)

print('✅ Features saved to data/processed/')